In [0]:
# Configures the data generator, storage paths, timezone, and reproducible random seed.

import json
import random

spark.conf.set("spark.sql.session.timeZone", "UTC")

SEED = 42
random.seed(SEED)

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"
VOLUME = "volume1"

BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

SHIPMENTS_PATH = f"{BASE}/landing/shipments"
PRODUCTS_PATH = f"{BASE}/landing/product_limits"
DELIVERIES_PATH = f"{BASE}/landing/delivery_events"

SENSOR_LANDING = f"{BASE}/landing/sensor_events"
SENSOR_STAGING = f"{BASE}/staging/sensor_events"

dbutils.fs.mkdirs(SENSOR_STAGING)
dbutils.fs.mkdirs(SENSOR_LANDING)

print("Data generator initialized")
print("Base path:", BASE)

### product generation

In [0]:
products = [
    ("P001", 2.0, 8.0),
    ("P002", 3.0, 7.0),
    ("P003", 1.0, 5.0)
]

products_df = spark.createDataFrame(
    products,
    """
    product_id STRING,
    min_temperature DOUBLE,
    max_temperature DOUBLE
    """
)

display(products_df)

### shipment generation

In [0]:
shipments = [
    (
        "SH001", "P001", "V001",
        "Pune", "Mumbai",
        "2026-10-07T08:00:00Z",
        "2026-10-07T12:00:00Z"
    ),
    (
        "SH002", "P002", "V002",
        "Mumbai", "Pune",
        "2026-10-07T08:30:00Z",
        "2026-10-07T13:00:00Z"
    ),
    (
        "SH003", "P003", "V003",
        "Pune", "Bengaluru",
        "2026-10-07T07:00:00Z",
        "2026-10-07T15:00:00Z"
    ),
    (
        "SH004", "P001", "V004",
        "Delhi", "Pune",
        "2026-10-07T06:00:00Z",
        "2026-10-07T14:00:00Z"
    ),
    (
        "SH005", "P002", "V005",
        "Hyderabad", "Chennai",
        "2026-10-07T09:00:00Z",
        "2026-10-07T16:00:00Z"
    )
]

shipments_df = spark.createDataFrame(
    shipments,
    """
    shipment_id STRING,
    product_id STRING,
    vehicle_id STRING,
    origin STRING,
    destination STRING,
    dispatch_time STRING,
    expected_arrival STRING
    """
)

display(shipments_df)


In [0]:
delivery_events = [
    # 30 min late
    (
        "D001",
        "SH001",
        "DELIVERED",
        "2026-10-07T12:30:00Z"
    ),

    # 10 min early
    (
        "D002",
        "SH002",
        "DELIVERED",
        "2026-10-07T12:50:00Z"
    ),

    # SH003 deliberately not delivered yet

    # exactly on time
    (
        "D004",
        "SH004",
        "DELIVERED",
        "2026-10-07T14:00:00Z"
    ),

    # 60 min late
    (
        "D005",
        "SH005",
        "DELIVERED",
        "2026-10-07T17:00:00Z"
    )
]

deliveries_df = spark.createDataFrame(
    delivery_events,
    """
    event_id STRING,
    shipment_id STRING,
    status STRING,
    event_time STRING
    """
)

display(deliveries_df)

In [0]:
# Generates sensor events from products_df + shipments_df (no hardcoding).
# Valid readings, boundary readings, temperature exceptions and bad records, all in one cell.
# Bad record types are distributed unevenly using weights.

from datetime import datetime, timezone
from collections import Counter

TOTAL_SENSOR_RECORDS = 10000
BAD_RECORD_RATE      = 0.01   # 1% -> quarantine
EXCEPTION_RATE       = 0.03   # 3% outside limits (still valid)
BOUNDARY_RATE        = 0.01   # 1% exactly on min/max (must be accepted)

# Uneven bad-record mix (weights, need not add to 1)
BAD_TYPE_WEIGHTS = {
    "invalid_temperature": 0.40,   # most common: sensor glitches
    "duplicate_event_id":  0.30,   # device resends
    "missing_event_id":    0.20,
    "unknown_shipment":    0.10,   # rarest
}

INVALID_TEMPS = ["hot", "N/A", "error", "--"]

rng = random.Random(SEED)

def parse_utc(s):
    return datetime.fromisoformat(s.replace("Z", "+00:00")).astimezone(timezone.utc)

def fmt(ts):
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")

# Reference data read from the DataFrames
limits = {
    r["product_id"]: (float(r["min_temperature"]), float(r["max_temperature"]))
    for r in products_df.collect()
}
shipment_rows      = shipments_df.collect()
valid_shipment_ids = {r["shipment_id"] for r in shipment_rows}

BAD_RECORD_COUNT  = round(TOTAL_SENSOR_RECORDS * BAD_RECORD_RATE)
GOOD_RECORD_COUNT = TOTAL_SENSOR_RECORDS - BAD_RECORD_COUNT
per_shipment, remainder = divmod(GOOD_RECORD_COUNT, len(shipment_rows))

# Good readings
good_rows, exception_count, boundary_count = [], 0, 0

for idx, s in enumerate(shipment_rows):
    shipment_id = s["shipment_id"]
    sensor_id   = "S" + shipment_id[2:]            # SH001 -> S001
    lo, hi      = limits[s["product_id"]]
    start       = parse_utc(s["dispatch_time"]).timestamp()
    end         = parse_utc(s["expected_arrival"]).timestamp()
    count       = per_shipment + (1 if idx < remainder else 0)

    for _ in range(count):
        ts = datetime.fromtimestamp(rng.uniform(start, end), tz=timezone.utc)
        r  = rng.random()
        if r < EXCEPTION_RATE:
            exception_count += 1
            temp = hi + rng.uniform(0.5, 5.0) if rng.random() < 0.5 else lo - rng.uniform(0.5, 3.0)
        elif r < EXCEPTION_RATE + BOUNDARY_RATE:
            boundary_count += 1
            temp = rng.choice([lo, hi])
        else:
            temp = rng.uniform(lo, hi)
        good_rows.append([None, shipment_id, sensor_id, fmt(ts), f"{temp:.2f}"])

good_rows.sort(key=lambda x: x[3])
for n, row in enumerate(good_rows, start=1):
    row[0] = f"E{n:06d}"

# Bad readings (weighted, uneven)
bad_type_list = rng.choices(
    population=list(BAD_TYPE_WEIGHTS.keys()),
    weights=list(BAD_TYPE_WEIGHTS.values()),
    k=BAD_RECORD_COUNT
)

bad_rows, bad_counts, next_id = [], Counter(), GOOD_RECORD_COUNT + 1

for bad_type in bad_type_list:
    t = rng.choice(good_rows)
    bad_counts[bad_type] += 1

    if bad_type == "missing_event_id":
        row = [None, t[1], t[2], t[3], t[4]]
    elif bad_type == "unknown_shipment":
        fake = f"SH{rng.randint(900, 999)}"
        row = [f"E{next_id:06d}", fake, "S" + fake[2:], t[3], t[4]]
        next_id += 1
    elif bad_type == "invalid_temperature":
        row = [f"E{next_id:06d}", t[1], t[2], t[3], rng.choice(INVALID_TEMPS)]
        next_id += 1
    else:  # duplicate_event_id -> exact resend
        row = list(t)
    bad_rows.append(row)

# Combine, sorted by time so bad rows land in every batch
sensor_events = [tuple(r) for r in sorted(good_rows + bad_rows, key=lambda x: x[3])]

assert len(sensor_events) == TOTAL_SENSOR_RECORDS

print("Total sensor events :", len(sensor_events))
print("Good readings       :", GOOD_RECORD_COUNT)
print("  temp exceptions   :", exception_count)
print("  boundary readings :", boundary_count)
print("Bad records         :", BAD_RECORD_COUNT)
for k, v in bad_counts.most_common():
    print(f"  {k:<20}: {v}")
print("Per shipment        :", dict(Counter(r[1] for r in good_rows)))

### batch simulation

In [0]:
# Splits the sensor events into configurable batches to simulate incremental file arrival.
try:
    dbutils.fs.rm(SENSOR_STAGING, recurse=True)
except Exception:
    pass
dbutils.fs.mkdirs(SENSOR_STAGING)

BATCH_SIZE = 1000

sensor_batches = [
    sensor_events[i:i + BATCH_SIZE]
    for i in range(0, len(sensor_events), BATCH_SIZE)
]

for index, batch in enumerate(sensor_batches, start=1):
    print(
        f"Batch {index}: {len(batch)} records"
    )

total_batched = sum(len(batch) for batch in sensor_batches)

assert total_batched == len(sensor_events), "Batching lost or duplicated records"

In [0]:
def write_sensor_batch(filename, records):

    output = []

    for row in records:

        record = {
            "event_id": row[0],
            "shipment_id": row[1],
            "sensor_id": row[2],
            "event_time": row[3],
            "temperature_c": row[4]
        }

        output.append(json.dumps(record))

    path = f"{SENSOR_STAGING}/{filename}"

    dbutils.fs.put(
        path,
        "\n".join(output) + "\n",
        overwrite=True
    )

    print(
        f"Created {filename} with {len(records)} records"
    )

### writing in the json format

In [0]:
# Writes every generated sensor batch without hardcoding individual batch filenames.

for index, batch in enumerate(sensor_batches, start=1):
    filename = f"batch_{index:03d}.json"

    write_sensor_batch(
        filename,
        batch
    )

In [0]:
display(
    dbutils.fs.ls(SENSOR_STAGING)
)


In [0]:
# Writes the product reference dataset into the landing directory.

(
    products_df
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(PRODUCTS_PATH)
)

product_files = dbutils.fs.ls(PRODUCTS_PATH)

print("Product data written successfully")
print("Product record count:", products_df.count())

display(product_files)

In [0]:
# Writes the shipment master dataset into the landing directory.

(
    shipments_df
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(SHIPMENTS_PATH)
)

shipment_files = dbutils.fs.ls(SHIPMENTS_PATH)

print("Shipment data written successfully")
print("Shipment record count:", shipments_df.count())

display(shipment_files)

In [0]:
# Writes the delivery-event dataset as JSON into the landing directory.

(
    deliveries_df
    .write
    .mode("overwrite")
    .json(DELIVERIES_PATH)
)

delivery_files = dbutils.fs.ls(DELIVERIES_PATH)

print("Delivery data written successfully")
print("Delivery event count:", deliveries_df.count())

display(delivery_files)

In [0]:
# Verifies all generated datasets and confirms the expected sensor records and batch files were created.

product_count = products_df.count()
shipment_count = shipments_df.count()
delivery_count = deliveries_df.count()

sensor_files = dbutils.fs.ls(SENSOR_STAGING)
sensor_file_count = len(sensor_files)
sensor_record_count = len(sensor_events)
batched_record_count = sum(len(batch) for batch in sensor_batches)

print("==============================================")
print("DATA GENERATION SUMMARY")
print("==============================================")
print("Products              :", product_count)
print("Shipments             :", shipment_count)
print("Delivery events       :", delivery_count)
print("Sensor events         :", sensor_record_count)
print("Batched sensor events :", batched_record_count)
print("Sensor batch files    :", sensor_file_count)
print("Expected batch files  :", len(sensor_batches))

print()
print("PRODUCT LANDING FILES")
display(dbutils.fs.ls(PRODUCTS_PATH))

print("SHIPMENT LANDING FILES")
display(dbutils.fs.ls(SHIPMENTS_PATH))

print("DELIVERY LANDING FILES")
display(dbutils.fs.ls(DELIVERIES_PATH))

print("SENSOR STAGING FILES")
display(sensor_files)

assert product_count == len(products), "Product record count mismatch"
assert shipment_count == len(shipments), "Shipment record count mismatch"
assert delivery_count == len(delivery_events), "Delivery record count mismatch"

assert sensor_record_count == TOTAL_SENSOR_RECORDS, (
    f"Expected {TOTAL_SENSOR_RECORDS} sensor records, found {sensor_record_count}"
)

assert batched_record_count == sensor_record_count, (
    "Sensor batching lost or duplicated records"
)

assert sensor_file_count == len(sensor_batches), (
    f"Expected {len(sensor_batches)} batch files, found {sensor_file_count}"
)

print()
print("==============================================")
print("DATA GENERATION STATUS: PASS")
print("==============================================")